# 実験概略
視差に対する微小斜視角の特性を分析する。

Schorモデルでは視差増加に対する線形的な微小斜視角の増加が予言される。<br>
融像の周波数応答を測定し、視差に対する微小斜視角の関数(以下Schor関数)を同定する。

このSchor関数が実際の微小斜視角に等しいか否かを検定する。<br>
検定してSchor関数と実斜視角が一致するならば、その結果はSchor理論を補強するものとなる。<br>
一致しないならば、融像の破れが生じない限りにおいて、<br>残差がsensory成分であるとみなしうる(Schor関数の分は眼球運動を反映しているから、motor成分である)。

# 実験プロトコル(案)
1. 大型弱視鏡やAPCT等により斜位(視)角を測定する($\theta_0$)
2. $\theta_0$を中和する(VRメガネの視標位置を調整して、プリズムに代える)
3. 正弦波掃引で視差を負荷し、融像の周波数応答を測定する。$a_f, k_f$を同定する。<br>Schor関数$D_S(\theta)$は次式に従う。
$$
D_S(\theta)=\frac{1}{1+a_f/k_f}\theta
$$
4. 線形掃引で視差を負荷し、それに伴い現れる(はずの)微小斜視角を測定する<br>
($D_{actual}$:実データ、$D_A$:真の融像関数、$\epsilon_A$:ノイズ)
$$
    D_{actual}(\theta) = D_A(\theta) + \epsilon_A
$$
5. $D_{actual}, a_f, k_f$をそれぞれ以降のセルに入力し、検定計算(および帰無仮説非棄却時はsensory成分の同定)を行う。


## 設定値入力GUI

以下のGUIで理論定数や実験条件を入力し、CSVをアップロードすると自動で再計算されます。

- `af`, `kf`: Schorモデル $D_S(\theta)=\frac{1}{1+a_f/k_f}\theta$ の定数（理論傾き = $\frac{1}{1+a_f/k_f}$、切片 = 0）
- `Alpha`: 有意水準


In [13]:
# Check and install required packages
required_packages = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'matplotlib': 'matplotlib',
    'scipy': 'scipy',
    'ipywidgets': 'ipywidgets',
}

import importlib
import subprocess
import sys
from io import BytesIO

for import_name, pip_name in required_packages.items():
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f"Installing {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])
        print(f"✓ {pip_name} installed")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
import ipywidgets as widgets
from IPython.display import display, Image

THEORY_AF = 1.0   # af デフォルト値
THEORY_KF = 2.0   # kf デフォルト値
ALPHA     = 0.05
POLY_DEG  = 2

loaded_data = None
loaded_file_name = None
output_area = widgets.Output()


def show_result(theory_af, theory_kf, alpha, df_data, file_name, poly_deg):
    df = df_data.copy()
    x_arr = df["disparity"].values
    y_arr = df["vergence error"].values

    # Schorモデルの理論傾き・切片
    theory_slope     = 1.0 / (1.0 + theory_af / theory_kf)
    theory_intercept = 0.0

    # --- 線形回帰 ---
    lin   = stats.linregress(x_arr, y_arr)
    a_hat = lin.slope
    b_hat = lin.intercept
    r2    = lin.rvalue ** 2

    n      = len(df)
    x_mean = x_arr.mean()
    sxx    = np.sum((x_arr - x_mean) ** 2)
    resid  = y_arr - (a_hat * x_arr + b_hat)
    s2     = np.sum(resid ** 2) / (n - 2)
    se_a   = np.sqrt(s2 / sxx)
    se_b   = np.sqrt(s2 * (1 / n + x_mean ** 2 / sxx))

    # H0: 傾き = 1/(1+af/kf),  H0: 切片 = 0
    t_a = (a_hat - theory_slope)     / se_a
    p_a = 2 * (1 - stats.t.cdf(abs(t_a), df=n - 2))
    t_b = (b_hat - theory_intercept) / se_b
    p_b = 2 * (1 - stats.t.cdf(abs(t_b), df=n - 2))

    # 推定傾き a_hat から af/kf を逆算
    # a_hat = 1/(1+af/kf) => af/kf = 1/a_hat - 1
    if np.isclose(a_hat, 0.0):
        af_over_kf_hat = np.inf
    else:
        af_over_kf_hat = (1.0 / a_hat) - 1.0

    # --- Schorモデルからの乖離 & 多項式近似 ---
    deviation   = y_arr - theory_slope * x_arr
    poly_coeffs = np.polyfit(x_arr, deviation, poly_deg)
    poly_fn     = np.poly1d(poly_coeffs)

    dev_fitted = poly_fn(x_arr)
    ss_res_dev = np.sum((deviation - dev_fitted) ** 2)
    ss_tot_dev = np.sum((deviation - deviation.mean()) ** 2)
    r2_dev     = 1 - ss_res_dev / ss_tot_dev if ss_tot_dev > 0 else float('nan')

    xx = np.linspace(x_arr.min(), x_arr.max(), 200)

    if np.isfinite(af_over_kf_hat):
        estimated_label = f"Estimated: y=1/(1+{af_over_kf_hat:.4g})*x"
    else:
        estimated_label = "Estimated: y=1/(1+inf)*x"

    # --- Figure 1: データ vs Schorモデル vs 推定直線 ---
    fig1, ax1 = plt.subplots(figsize=(8, 5))
    ax1.scatter(x_arr, y_arr, alpha=0.7, label="Experimental data")
    ax1.plot(
        xx,
        theory_slope * xx,
        "--",
        linewidth=2,
        label=f"Schor model: y=1/(1+{theory_af/theory_kf:.4g})*x",
    )
    ax1.plot(xx, a_hat * xx + b_hat, linewidth=2, label=estimated_label)
    ax1.set_xlabel("disparity θ")
    ax1.set_ylabel("vergence error D")
    ax1.legend()
    ax1.grid(alpha=0.3)
    plt.tight_layout()
    buf1 = BytesIO()
    fig1.savefig(buf1, format='png', dpi=100)
    plt.close(fig1)
    buf1.seek(0)
    png1 = buf1.read()

    # --- Figure 2: Schorモデルからの乖離 & 多項式近似 ---
    fig2, ax2 = plt.subplots(figsize=(8, 5))
    ax2.scatter(x_arr, deviation, alpha=0.7, color='C1', label="Deviation from Schor model")
    ax2.plot(xx, poly_fn(xx), linewidth=2, color='C3',
             label=f"Poly fit (deg={poly_deg})")
    ax2.axhline(0, color='gray', linestyle=':', linewidth=1)
    ax2.set_xlabel("disparity θ")
    ax2.set_ylabel("D_actual - D_S(θ)")
    ax2.legend()
    ax2.grid(alpha=0.3)
    plt.tight_layout()
    buf2 = BytesIO()
    fig2.savefig(buf2, format='png', dpi=100)
    plt.close(fig2)
    buf2.seek(0)
    png2 = buf2.read()

    # --- 出力 ---
    output_area.clear_output(wait=True)
    with output_area:
        print(f"ファイル: {file_name}  (n={n})")
        print(f"Schorモデル: D_S(θ) = 1/(1+af/kf)*θ  [af={theory_af:.4g}, kf={theory_kf:.4g}]")
        print(f"  理論傾き = {theory_slope:.6f},  理論切片 = 0")
        print()
        print("=== 推定結果 ===")
        print(f"推定 af/kf      = {af_over_kf_hat:.6f}")
        print(f"推定切片 (b_hat) = {b_hat:.4f}")
        print(f"R²               = {r2:.4f}")
        print()
        print("=== 理論値との検定結果（両側t検定） ===")
        print(f"H0: 傾き = 1/(1+af/kf) = {theory_slope:.4f} | t={t_a:.4f}, p={p_a:.4g}")
        print(f"H0: 切片 = 0            | t={t_b:.4f}, p={p_b:.4g}")
        print()
        if p_a >= alpha and p_b >= alpha:
            print(f"結論: 有意水準 {alpha} で理論との有意差は認められず、Schorモデルは妥当とみなせる。")
        else:
            print(f"結論: 有意水準 {alpha} で理論との差が示唆され、Schorモデルの妥当性に課題がある可能性。")
        display(Image(png1))

        print()
        print(f"=== 乖離成分 (D_actual - D_S) の多項式近似 (次数={poly_deg}) ===")
        print(f"係数 (高次→定数): {np.array2string(poly_coeffs, precision=6)}")
        print(f"R² (poly fit)    = {r2_dev:.4f}")
        display(Image(png2))


def run_analysis():
    if loaded_data is None:
        return
    show_result(
        theory_af=af_widget.value,
        theory_kf=kf_widget.value,
        alpha=alpha_widget.value,
        df_data=loaded_data,
        file_name=loaded_file_name,
        poly_deg=poly_deg_widget.value,
    )


def _set_status(text, color='#333'):
    status_html.value = f"<span style='color:{color};'>{text}</span>"


def _extract_uploaded_file(uploaded):
    # ipywidgets 7/8 の差異（dict, tuple/list, UploadedFile）を吸収
    if isinstance(uploaded, dict):
        first_key = next(iter(uploaded))
        item = uploaded[first_key]
    else:
        item = uploaded[0]

    if isinstance(item, dict):
        file_name = item.get('name', 'uploaded.csv')
        file_content = item.get('content', b'')
    else:
        file_name = getattr(item, 'name', 'uploaded.csv')
        file_content = getattr(item, 'content', b'')

    return file_name, bytes(file_content)


def _load_uploaded_csv(uploaded):
    global loaded_data, loaded_file_name

    if not uploaded:
        _set_status('⚠ CSVが選択されていません。', '#b26a00')
        return

    file_name, file_content = _extract_uploaded_file(uploaded)
    df = pd.read_csv(BytesIO(file_content))
    df.columns = [str(c).replace('\ufeff', '').strip() for c in df.columns]

    if 'disparity' not in df.columns or 'vergence error' not in df.columns:
        output_area.clear_output(wait=True)
        with output_area:
            print('❌ CSVには "disparity" と "vergence error" 列が必要です')
            print(f"  検出された列: {list(df.columns)}")
        _set_status('❌ CSV列名が不正です。', '#c62828')
        return

    loaded_data = df
    loaded_file_name = file_name
    _set_status(f"✅ {file_name} を読み込みました。", '#2e7d32')
    run_analysis()


def _reset_upload_widget():
    # 次回アップロードのために値をクリア（互換性重視）
    try:
        if hasattr(file_upload_widget.value, 'clear'):
            file_upload_widget.value.clear()
        file_upload_widget._counter = 0
    except Exception:
        pass


def on_file_uploaded(change):
    try:
        uploaded = change.get('new') if isinstance(change, dict) and 'new' in change else file_upload_widget.value
        _load_uploaded_csv(uploaded)
        _reset_upload_widget()
    except Exception as e:
        output_area.clear_output(wait=True)
        with output_area:
            print(f"❌ エラー: {e}")
        _set_status('❌ 読み込み時エラーが発生しました。', '#c62828')


def on_manual_load_clicked(_):
    on_file_uploaded({'new': file_upload_widget.value})


def on_poly_deg_changed(change):
    run_analysis()


# ウィジェット定義
af_widget    = widgets.FloatText(value=THEORY_AF, description="af")
kf_widget    = widgets.FloatText(value=THEORY_KF, description="kf")
alpha_widget = widgets.BoundedFloatText(value=ALPHA, min=0.0001, max=0.5,
                                        step=0.001, description="Alpha")
poly_deg_widget = widgets.BoundedIntText(value=POLY_DEG, min=1, max=20,
                                          step=1, description="Poly deg")

file_upload_widget = widgets.FileUpload(accept='.csv', multiple=False,
                                        description='CSVをアップロード')
manual_load_button = widgets.Button(description='手動で読み込み', button_style='info')
status_html = widgets.HTML(value='')
file_upload_widget.observe(on_file_uploaded, names='value')
manual_load_button.on_click(on_manual_load_clicked)
poly_deg_widget.observe(on_poly_deg_changed, names='value')

display(widgets.VBox([
    widgets.HTML("<b>Schorモデルパラメータ設定</b> &nbsp; D_S(θ) = 1 / (1 + af/kf) * θ"),
    widgets.HBox([af_widget, kf_widget, alpha_widget]),
    widgets.HTML("<b>乖離成分の多項式近似</b>"),
    widgets.HBox([poly_deg_widget]),
    widgets.HTML("<b>データロード</b>"),
    widgets.HTML("CSVアップロード後に自動計算されます。反応しない場合は手動で読み込みを押してください。"),
    widgets.HBox([file_upload_widget, manual_load_button]),
    status_html,
    output_area,
]))
